# LOKLOK Pilot: Gemini Persona Agents vs the Rule-Based ABM

This notebook runs a **small, transparent pilot comparison**. It asks a Gemini model to appraise the end state of the same measured persona under the eight A/B/C conditions, then compares the resulting ratings with the final rule-based ABM outputs.

**What this is**
- A single-shot LLM appraisal proxy per persona and condition.
- A descriptive comparison of levels, G0-to-G1 contrasts, and the rank order of the eight SR_eval cells.
- A reusable pilot harness with response caching and a machine-readable output packet.

**What this is not**
- It is not human data.
- It is not an 18-session LLM simulation; the LLM receives the condition description and returns an end-of-study appraisal.
- It is not a replacement for the rule-based ABM or for a field experiment.

**Inputs**
- A small `colab_gemini_bundle.zip` is downloaded from the tunnel and extracted; no file picker is used.
- It contains the measured matrix, the final ABM output, the factorial cell table, and the persona deltas.

**Outputs**
- `gemini_responses.csv`, `gemini_abm_comparison.csv`, `gemini_abm_summary.json`, `gemini_abm_report.md`, and figures.

## 1. Install the Gemini SDK

In [ ]:
!pip -q install google-genai pandas numpy scipy matplotlib

## 2. Configure the pilot

In [ ]:
import os, json, time, getpass, re, zipfile, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 20261105
N_PERSONAS = 24
N_SESSIONS = 18
MODEL = os.environ.get('GEMINI_MODEL', 'gemini-2.5-flash')
TEMPERATURE = 0.2
FORCE_RERUN = False
PROMPT_VERSION = 'v1'
CELLS = [(0, 0, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1),
         (1, 1, 0), (1, 0, 1), (0, 1, 1), (1, 1, 1)]
print('model:', MODEL, '| personas:', N_PERSONAS, '| calls:', N_PERSONAS * len(CELLS))

## 3. Load the key and the input files

In [ ]:
def _secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return None

API_KEY = _secret('GEMINI_API_KEY') or _secret('GOOGLE_API_KEY')
if not API_KEY:
    raise RuntimeError('No GEMINI_API_KEY found. Add it to Colab Secrets (key icon) as GEMINI_API_KEY, then reconnect and rerun.')

BUNDLE_URL = 'https://cycles-bless-equal-printers.trycloudflare.com/colab_gemini_bundle.zip'
BUNDLE_SHA256 = '0f6521bed1fac11d60aab35c5ac943cddc75c60a138362b0407688ca6e07225d'
BASE = Path('/content') if Path('/content').exists() else Path.cwd()
ZIP_PATH = BASE / 'colab_gemini_bundle.zip'
ROOT = BASE / 'loklok_gemini'
if not (ROOT / '03_data' / 'measured' / 'respondents_52.csv').exists():
    print('Downloading bundle...', BUNDLE_URL)
    import urllib.request
    urllib.request.urlretrieve(BUNDLE_URL, ZIP_PATH)
    digest = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
    assert digest == BUNDLE_SHA256, f'bundle hash mismatch: {digest}'
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(BASE)
print('bundle root:', ROOT)
assert (ROOT / '03_data' / 'measured' / 'respondents_52.csv').exists()

def _find(name, extra):
    for p in [ROOT / name, ROOT / '03_data' / 'measured' / name,
              ROOT / '03_data' / 'model_outputs' / name] + [ROOT / e for e in extra]:
        if p.exists():
            return p
    raise FileNotFoundError(name)

RESP_PATH = _find('respondents_52.csv', [])
RESULTS_PATH = _find('pipeline_results.json', [])
print('respondents:', RESP_PATH)
print('model output:', RESULTS_PATH)

## 4. Build persona cards and condition text

In [ ]:
resp = pd.read_csv(RESP_PATH)
resp.columns = [c.strip() for c in resp.columns]
pool = resp.reset_index(drop=True)
n_take = min(N_PERSONAS, len(pool))
sample = pool.sample(n=n_take, random_state=SEED).reset_index(drop=True)

def _num(value, default):
    try:
        value = float(value)
        return value if np.isfinite(value) else default
    except Exception:
        return default

def persona_card(row):
    pb = _num(row.get('PB_mean', row.get('PB1', 4.0)), 4.0)
    star = _num(row.get('star_rating', 3.5), 3.5)
    duration = _num(row.get('ride_duration_minutes', 15), 15)
    age = _num(row.get('age', 20), 20)
    return (
        f'Age: {age:.0f}; gender: {row.get("gender", "unspecified")}; '
        f'academic year: {row.get("academic_year", "unspecified")}; '
        f'faculty: {row.get("faculty", "unspecified")}; '
        f'ride context: {row.get("context", "unspecified")}; '
        f'baseline platform-benefit rating: {pb:.1f}/7; '
        f'baseline star rating: {star:.1f}/5; '
        f'mood: {row.get("student_mood", "unspecified")}; '
        f'interaction level: {row.get("interaction_level", "unspecified")}; '
        f'ride duration: {duration:.0f} minutes.'
    )

A_ON = ('A on: BRS credibility architecture with source lineage, credibility-weighted ranking, '
        'a confidence badge, and visible reasons for each recommendation.')
A_OFF = ('A off: conventional opaque engagement-optimized ranking, with no source-lineage display '
         'and no credibility badge.')
B_ON = ('B on: MORE diversity-aware ranking plus Hilbert diversity and Echo Meter cooling when '
        'the repetition index reaches 0.75.')
B_OFF = ('B off: no diversity constraint and no cooling; the feed keeps optimizing immediate '
         'engagement.')
C_ON = ('C on: ERIC moderation plus short EMA reflection prompts that ask the user to evaluate '
        'source quality and their own reasoning.')
C_OFF = ('C off: no moderation layer and no reflective prompts.')

def condition_text(a, b, c):
    return '\n'.join([A_ON if a else A_OFF, B_ON if b else B_OFF, C_ON if c else C_OFF])

PROMPT = '''You are an LLM persona agent in a methodological pilot. Adopt the persona below, but do not claim to be a real student and do not add facts about the person. You are not asked to simulate each session; give the end-of-study appraisal implied by the persona and the condition.\n\nPersona:\n__CARD__\n\nCondition after __SESSIONS__ short listening sessions:\n__CONDITION__\n\nEstimate the terminal ratings for this persona. Use the full scales, not school-grade conventions.\n- sr_eval: evaluative Student Rating, 1-7 (1 very negative, 4 neutral, 7 very positive).\n- sr_comp: competence-format media-literacy rating, 0-100.\n- engagement: raw engagement, 0-1.\n- confidence: your confidence in this appraisal, 0-1.\n\nReturn only one JSON object with exactly these keys:\n{"sr_eval": 0, "sr_comp": 0, "engagement": 0, "confidence": 0, "rationale": "one short sentence"}'''


## 5. Call the API, validate responses, and cache

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=API_KEY)
CACHE_PATH = Path('/content/loklok_gemini_cache.jsonl') if Path('/content').exists() else Path.cwd() / 'loklok_gemini_cache.jsonl'
OUTDIR = Path('/content/loklok_gemini_vs_abm') if Path('/content').exists() else Path.cwd() / 'loklok_gemini_vs_abm'
OUTDIR.mkdir(parents=True, exist_ok=True)

def parse_response(text):
    match = re.search(r'\{.*\}', text or '', re.S)
    if not match:
        raise ValueError('no JSON object in response')
    obj = json.loads(match.group(0))
    out = {}
    for key in ['sr_eval', 'sr_comp', 'engagement', 'confidence']:
        out[key] = float(obj[key])
    out['rationale'] = str(obj.get('rationale', ''))[:300]
    out['sr_eval'] = min(7.0, max(1.0, out['sr_eval']))
    out['sr_comp'] = min(100.0, max(0.0, out['sr_comp']))
    out['engagement'] = min(1.0, max(0.0, out['engagement']))
    out['confidence'] = min(1.0, max(0.0, out['confidence']))
    return out

def call_agent(prompt):
    last = None
    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    top_p=0.9,
                    response_mime_type='application/json',
                ),
            )
            return parse_response(getattr(response, 'text', str(response)))
        except Exception as exc:
            last = exc
            wait = min(20, 2 ** attempt)
            print(f'retry {attempt + 1}/5 in {wait}s: {type(exc).__name__}: {str(exc)[:180]}')
            time.sleep(wait)
    raise RuntimeError(f'Gemini call failed: {last}')

def load_cache():
    cache = {}
    if CACHE_PATH.exists():
        for line in CACHE_PATH.read_text(encoding='utf-8').splitlines():
            try:
                rec = json.loads(line)
                cache[rec['run_key']] = rec
            except Exception:
                continue
    return cache

def append_cache(record):
    with open(CACHE_PATH, 'a', encoding='utf-8') as fh:
        fh.write(json.dumps(record, ensure_ascii=False) + '\n')

## 6. Run the pilot

In [ ]:
cache = load_cache()
records = {}
total = len(sample) * len(CELLS)
done = 0
for persona_idx, row in sample.iterrows():
    card = persona_card(row)
    persona_key = f'P{persona_idx + 1:03d}'
    for (a, b, c) in CELLS:
        cell_key = f'{a}{b}{c}'
        prompt = (PROMPT.replace('__CARD__', card)
                        .replace('__SESSIONS__', str(N_SESSIONS))
                        .replace('__CONDITION__', condition_text(a, b, c)))
        prompt_hash = hashlib.sha256(prompt.encode('utf-8')).hexdigest()[:16]
        run_key = f'{persona_key}|{cell_key}|{MODEL}|{PROMPT_VERSION}|{prompt_hash}'
        if not FORCE_RERUN and run_key in cache:
            records[run_key] = cache[run_key]
            done += 1
            continue
        result = call_agent(prompt)
        record = {
            'run_key': run_key, 'persona_key': persona_key, 'cell_key': cell_key,
            'A': a, 'B': b, 'C': c, 'model': MODEL, 'prompt_version': PROMPT_VERSION,
            'prompt_sha256': prompt_hash, **result,
        }
        records[run_key] = record
        append_cache(record)
        done += 1
        if done % 8 == 0 or done == total:
            print(f'{done}/{total} calls complete')
            pd.DataFrame(records.values()).to_csv(OUTDIR / 'gemini_responses_partial.csv', index=False)

responses = pd.DataFrame(records.values())
responses.to_csv(OUTDIR / 'gemini_responses.csv', index=False)
print(responses.groupby('cell_key')[['sr_eval', 'sr_comp', 'engagement']].mean().round(3))

## 7. Compare with the rule-based ABM

In [ ]:
pipeline = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
fa = pipeline['factorial']
abm_cells = pd.DataFrame(fa['cell_means'])
abm_cells['cell_key'] = abm_cells['factors'].apply(lambda x: ''.join(str(int(v)) for v in x))
abm_sr = abm_cells.set_index('cell_key')['mean']
llm_cell = responses.groupby('cell_key')[['sr_eval', 'sr_comp', 'engagement']].mean()

def boot_ci(values, n_boot=5000, seed=SEED):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return [float('nan'), float('nan')]
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(n_boot, len(values)))
    means = values[idx].mean(axis=1)
    return [float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]

def endpoint(metric, abm_g0, abm_g1):
    pivot = responses.pivot_table(index='persona_key', columns='cell_key', values=metric)
    delta = (pivot['111'] - pivot['000']).dropna() if {'111', '000'}.issubset(pivot.columns) else pd.Series(dtype=float)
    llm_g0 = float(llm_cell.loc['000', metric])
    llm_g1 = float(llm_cell.loc['111', metric])
    return {
        'llm_g0': llm_g0, 'llm_g1': llm_g1, 'llm_delta': llm_g1 - llm_g0,
        'abm_g0': float(abm_g0), 'abm_g1': float(abm_g1),
        'abm_delta': float(abm_g1 - abm_g0),
        'difference_in_delta': float((llm_g1 - llm_g0) - (abm_g1 - abm_g0)),
        'llm_persona_delta_mean': float(delta.mean()) if len(delta) else float('nan'),
        'llm_persona_delta_ci95': boot_ci(delta) if len(delta) else [float('nan'), float('nan')],
    }

endpoints = {
    'sr_eval': endpoint('sr_eval', fa['sr_eval_g0'], fa['sr_eval_g1']),
    'sr_comp': endpoint('sr_comp', fa['sr_comp_g0'], fa['sr_comp_g1']),
    'engagement': endpoint('engagement', fa['engagement_g0'], fa['engagement_g1']),
}

cells_compare = pd.concat([llm_cell['sr_eval'].rename('llm'), abm_sr.rename('abm')], axis=1).dropna()
cells_compare['difference'] = cells_compare['llm'] - cells_compare['abm']
cell_stats = {
    'n_cells': int(len(cells_compare)),
    'mae': float(np.mean(np.abs(cells_compare['difference']))),
    'rmse': float(np.sqrt(np.mean(cells_compare['difference'] ** 2))),
    'pearson_r': float(stats.pearsonr(cells_compare['llm'], cells_compare['abm'])[0]) if len(cells_compare) > 2 else float('nan'),
    'spearman_rho': float(stats.spearmanr(cells_compare['llm'], cells_compare['abm'])[0]) if len(cells_compare) > 2 else float('nan'),
}

summary = {
    'model': MODEL, 'prompt_version': PROMPT_VERSION, 'seed': SEED,
    'n_personas': int(responses['persona_key'].nunique()),
    'n_calls': int(len(responses)), 'n_sessions_described': N_SESSIONS,
    'endpoints': endpoints, 'cell_level_sr_eval': cell_stats,
    'cell_means_sr_eval': cells_compare.round(4).to_dict(orient='index'),
    'integrity': 'LLM pilot only; not human data and not an 18-session simulation.',
}
(OUTDIR / 'gemini_abm_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
cells_compare.round(4).to_csv(OUTDIR / 'gemini_abm_comparison.csv')
print(json.dumps(summary, indent=2))

## 8. Report and figures

In [ ]:
lines = [
    '# Gemini Persona Agents vs Rule-Based ABM: Pilot Comparison', '',
    f"Model: `{MODEL}`; prompt version: `{PROMPT_VERSION}`; personas: {summary['n_personas']}; calls: {summary['n_calls']}.", '',
    '**Integrity:** This is an LLM appraisal pilot, not human data and not an 18-session simulation. The ABM baseline is the final rule-based model.', '',
    '## Endpoint comparison', '',
    '| Metric | LLM G0 | LLM G1 | LLM delta | ABM delta | LLM - ABM delta |',
    '|---|---:|---:|---:|---:|---:|',
]
labels = {'sr_eval': 'Evaluative SR (1-7)', 'sr_comp': 'Competence SR (0-100)', 'engagement': 'Engagement (0-1)'}
for metric, label in labels.items():
    e = endpoints[metric]
    lines.append(f"| {label} | {e['llm_g0']:.3f} | {e['llm_g1']:.3f} | {e['llm_delta']:+.3f} | {e['abm_delta']:+.3f} | {e['difference_in_delta']:+.3f} |")
lines += [
    '', '## Cell-level SR_eval comparison', '',
    f"Eight-cell MAE: **{cell_stats['mae']:.3f}**; RMSE: **{cell_stats['rmse']:.3f}**; Pearson r: **{cell_stats['pearson_r']:.3f}**; Spearman rho: **{cell_stats['spearman_rho']:.3f}**.", '',
    'The eight-cell correlation is descriptive only: with eight cells it has very low power.', '',
    '## Persona-level G1 - G0', '',
    '| Metric | Mean | 95% bootstrap interval |', '|---|---:|---:|',
]
for metric, label in labels.items():
    e = endpoints[metric]
    lines.append(f"| {label} | {e['llm_persona_delta_mean']:+.3f} | [{e['llm_persona_delta_ci95'][0]:+.3f}, {e['llm_persona_delta_ci95'][1]:+.3f}] |")
lines += [
    '', '## Reading the result', '',
    '- Agreement in the sign of G1 - G0 is more informative than exact point values.',
    '- A large level difference can reflect the LLM\'s scale habits; the delta and rank comparisons are the primary checks.',
    '- Any disagreement is a finding about the LLM proxy, not evidence against the rule-based model or against human students.', '',
    '## Guardrails', '',
    '- Do not present these outputs as measured human ratings.',
    '- Do not infer mechanism from a single-shot appraisal.',
    '- Report the model, prompt version, seed, number of calls, and any failures.',
]
(OUTDIR / 'gemini_abm_report.md').write_text('\n'.join(lines), encoding='utf-8')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (metric, label) in zip(axes, labels.items()):
    e = endpoints[metric]
    ax.bar(['G0', 'G1'], [e['llm_g0'], e['llm_g1']], color=['#9fb7c9', '#2f6f8f'], width=0.55, label='Gemini')
    ax.scatter(['G0', 'G1'], [e['abm_g0'], e['abm_g1']], color='#c44e52', zorder=3, label='ABM')
    ax.set_title(label)
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTDIR / 'gemini_abm_endpoints.png', dpi=220)
plt.show()

fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.scatter(cells_compare['abm'], cells_compare['llm'], color='#2f6f8f')
for key, row in cells_compare.iterrows():
    ax.annotate(key, (row['abm'], row['llm']), fontsize=8, xytext=(3, 3), textcoords='offset points')
lo = min(cells_compare['abm'].min(), cells_compare['llm'].min()) - 0.1
hi = max(cells_compare['abm'].max(), cells_compare['llm'].max()) + 0.1
ax.plot([lo, hi], [lo, hi], color='0.65', lw=1)
ax.set_xlabel('Rule-based ABM cell mean')
ax.set_ylabel('Gemini cell mean')
ax.set_title('SR_eval: eight A/B/C cells')
fig.tight_layout()
fig.savefig(OUTDIR / 'gemini_abm_cells.png', dpi=220)
plt.show()
print('\n'.join(lines[:18]))

## 9. Package the output

In [ ]:
zip_path = OUTDIR / 'loklok_gemini_vs_abm_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTDIR.glob('*'):
        if p.is_file() and p.name != zip_path.name:
            z.write(p, arcname=p.name)
print('wrote', zip_path, zip_path.stat().st_size, 'bytes')
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print('Download skipped:', exc)

## 10. What to report in the thesis or defense

Use the pilot as a **methodological comparison**, not as a second empirical study. Report:

1. the model and prompt version;
2. the number of personas and calls;
3. the endpoint G0/G1 contrasts for both arms;
4. the eight-cell SR_eval agreement (MAE, RMSE, rank correlation);
5. the direction of disagreement and the single-shot limitation.

A defensible sentence is: *“In a pilot comparison, a Gemini persona agent reproduced the sign of the evaluative-rating contrast but diverged in level and in some cell rankings; this is evidence about the LLM proxy, not about human students.”* Only use that sentence if the output actually supports it.